# TruthLens — Binary Human-Face Authenticity Baseline Training (Google Colab GPU)

Trains the first working human-face authenticity detector on 140,000 face images:
- **Class 0 (`genuine`):** 70,000 Flickr-Faces-HQ (FFHQ) photographic human portraits
- **Class 1 (`ai_generated`):** 70,000 StyleGAN-generated synthetic human faces

> **Important Scope Note:** This binary baseline is optimized to detect whole-face GAN synthesis (FFHQ vs StyleGAN). It is not trained to detect face swaps, facial reenactments, or all modern diffusion generators. Those will be added in subsequent multi-class stages.

### Pipeline Highlights
- **Backbone:** ImageNet-pretrained `efficientnet_b0` (`timm==1.0.3`) with 2-class head (`Dropout(0.2)` + `Linear(1280, 2)`).
- **Two-Stage Fine-Tuning:**
  - **Stage 1 (Warmup):** Head only trainable; backbone frozen in `eval()` mode.
  - **Stage 2 (Fine-tuning):** Selective unfreeze of upper blocks (`blocks.5`, `blocks.6`, `conv_head`, `bn2`, `head`) with differential learning rates.
- **Checkpoint Selection:** Best model selected strictly by **Validation Macro-F1**.
- **Outputs to Google Drive:** Saves deployment checkpoint (`human_face_efficientnet_b0_best.pt`), resume state, training curves, confusion matrix, ROC-AUC, and test evaluation reports.

## Step 1: Check GPU Runtime & Environment
Make sure Google Colab is running with a GPU runtime: **Runtime > Change runtime type > T4 GPU** (or A100/V100).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")

## Step 2: Install / Verify `timm==1.0.3` and Dependencies
Ensures `timm==1.0.3` is installed without breaking existing Colab libraries.

In [ ]:
import importlib.util
import subprocess
import sys

try:
    import timm
    if timm.__version__ != "1.0.3":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==1.0.3"])
        import timm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==1.0.3"])
    import timm

print(f"Using timm=={timm.__version__}")

## Step 3: Dataset Ingestion (Google Drive Mount & Fast NVMe Extraction)

### Upload Instructions:
1. **Prepare Dataset on Local Machine:**
   The dataset is located at `ai-ml/data/deepfake_detection/`.
   Create a zip archive (PowerShell):
   ```powershell
   Compress-Archive -Path "ai-ml\data\deepfake_detection\*" -DestinationPath "deepfake_detection.zip"
   ```
2. **Upload to Google Drive:**
   Upload `deepfake_detection.zip` to your Google Drive folder:
   `MyDrive/TruthLens/deepfake_detection.zip`
3. **Automatic Fast NVMe Extraction:**
   The cell below unpacks the zip file directly to local Colab SSD (`/content/human_face_data`) in ~1 minute, ensuring blazing-fast data loading during GPU training!
   *(If you prefer mounting an unzipped folder, set `OPTIONAL_ARCHIVE_PATH = None` and point `DRIVE_ROOT` to the folder).*

In [ ]:
import json
import tarfile
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# --- PATH CONFIGURATION ---
DRIVE_ROOT = Path("/content/drive/MyDrive/TruthLens")
OUTPUT_DIR = DRIVE_ROOT / "checkpoints" / "hf_binary_effnet_b0_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional ZIP/TAR archive on Google Drive for fast local SSD training (RECOMMENDED):
OPTIONAL_ARCHIVE_PATH = DRIVE_ROOT / "deepfake_detection.zip"
LOCAL_EXTRACT_DIR = Path("/content/human_face_data")

def extract_dataset_archive(archive_path: Path, target_dir: Path) -> Path:
    """Extract .zip or .tar/.tar.gz/.tgz dataset archive cleanly to `target_dir`."""
    archive_path = Path(archive_path)
    target_dir = Path(target_dir)
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    target_dir.mkdir(parents=True, exist_ok=True)
    lower_name = archive_path.name.lower()
    if lower_name.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(target_dir)
    elif lower_name.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2")):
        with tarfile.open(archive_path, "r:*") as tf:
            tf.extractall(target_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path.name}. Use .zip or .tar/.tar.gz.")
    return target_dir

def resolve_extracted_data_root(base_dir: Path, manifest_data: dict) -> Path:
    """Auto-detect root directory matching relative paths stored in manifest or directory structure."""
    images = manifest_data.get("images", [])
    if images:
        sample_rel = Path(images[0]["image_path"])
        if sample_rel.is_absolute():
            raise ValueError(
                f"Manifest contains non-portable absolute image_path '{sample_rel}'. "
                "Regenerate locally with `--materialize` or a single relative dataset root."
            )
        if (base_dir / sample_rel).exists():
            return base_dir
        for sub in sorted(base_dir.iterdir()):
            if sub.is_dir() and (sub / sample_rel).exists():
                return sub
            if sub.is_dir():
                for nested in sorted(sub.iterdir()):
                    if nested.is_dir() and (nested / sample_rel).exists():
                        return nested
        raise FileNotFoundError(
            f"Could not locate '{sample_rel}' under '{base_dir}' or its immediate subfolders."
        )
    for cand in [base_dir] + [s for s in base_dir.iterdir() if s.is_dir()]:
        if (cand / "train" / "real").exists() or (cand / "train" / "genuine").exists():
            return cand
    return base_dir

if OPTIONAL_ARCHIVE_PATH is not None and Path(OPTIONAL_ARCHIVE_PATH).exists():
    print(f"Extracting dataset archive from: {OPTIONAL_ARCHIVE_PATH}")
    CANDIDATE_ROOT = extract_dataset_archive(Path(OPTIONAL_ARCHIVE_PATH), LOCAL_EXTRACT_DIR)
else:
    candidates = [
        DRIVE_ROOT / "deepfake_detection",
        DRIVE_ROOT / "human_face",
        DRIVE_ROOT,
        LOCAL_EXTRACT_DIR,
    ]
    CANDIDATE_ROOT = next((c for c in candidates if c.exists()), DRIVE_ROOT)

MANIFEST_CANDIDATES = [
    CANDIDATE_ROOT / "split_manifest.json",
    CANDIDATE_ROOT / "dataset_manifest.json",
    DRIVE_ROOT / "split_manifest.json",
    DRIVE_ROOT / "dataset_manifest.json",
]
MANIFEST_PATH = next((m for m in MANIFEST_CANDIDATES if m.exists()), None)
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH else {}

DATA_ROOT = resolve_extracted_data_root(CANDIDATE_ROOT, MANIFEST)
print(f"Data Root:  {DATA_ROOT}")
print(f"Manifest:   {MANIFEST_PATH}")
print(f"Output Dir: {OUTPUT_DIR}")

## Step 4: Binary Training Configuration & Reproducibility
Sets up seeds, batch size, mixed precision (AMP), and differential learning rates for two-stage transfer learning.

In [ ]:
import random
import numpy as np
import torch

CONFIG = {
    "model_version": "human_face_binary_effnet_b0_v1",
    "backbone": "efficientnet_b0",
    "num_classes": 2,
    "class_names": ["genuine", "ai_generated"],
    "class_to_id": {"genuine": 0, "ai_generated": 1},
    "input_size": 224,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
    "seed": 42,
    "batch_size": 64,
    "num_workers": 2,
    "stage1_epochs": 3,
    "stage2_epochs": 10,
    "stage1_lr": 3e-4,
    "stage2_backbone_lr": 3e-5,
    "stage2_head_lr": 1e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
    "use_amp": True,
    "early_stopping_patience": 5,
    "max_corrupt_fraction": 0.01,
    "resume_if_available": True,
}

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Configured model_version={CONFIG['model_version']} on device={DEVICE}")

## Step 5: Dataset Loading, Split Validation, and Safe Streaming Loaders
Loads the existing `train` (112,000), `val` (14,000), and `test` (14,000) splits directly from `split_manifest.json` or directory structure, verifying zero cross-split leakage.

In [ ]:
import hashlib
import logging
from collections import Counter
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

logger = logging.getLogger("truthlens.colab")

records_by_split = {"train": [], "val": [], "test": []}
sha_by_split = {"train": set(), "val": set(), "test": set()}

if MANIFEST and "images" in MANIFEST:
    for rec in MANIFEST["images"]:
        sp = rec["split"]
        records_by_split[sp].append(rec)
        if rec.get("sha256"):
            sha_by_split[sp].add(rec["sha256"])
elif MANIFEST and "splits" in MANIFEST:
    for rel_path, sp in MANIFEST["splits"].items():
        is_real = "real" in rel_path.lower()
        lbl = 0 if is_real else 1
        cname = "genuine" if is_real else "ai_generated"
        subset = "ffhq" if is_real else "stylegan"
        if (DATA_ROOT / sp / rel_path).exists():
            full_rel = f"{sp}/{rel_path}"
        else:
            full_rel = rel_path
        records_by_split[sp].append({
            "image_path": full_rel,
            "split": sp,
            "label": lbl,
            "class_name": cname,
            "source_subset": subset,
        })
else:
    for sp in ("train", "val", "test"):
        for sub, lbl, cname, subset in [
            ("real", 0, "genuine", "ffhq"),
            ("genuine", 0, "genuine", "ffhq"),
            ("fake", 1, "ai_generated", "stylegan"),
            ("ai_generated", 1, "ai_generated", "stylegan"),
        ]:
            folder = DATA_ROOT / sp / sub
            if folder.exists():
                for f in sorted(folder.glob("*.*")):
                    if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
                        records_by_split[sp].append({
                            "image_path": f"{sp}/{sub}/{f.name}",
                            "split": sp,
                            "label": lbl,
                            "class_name": cname,
                            "source_subset": subset,
                        })

def validate_split_classes(split_records: dict, class_names: list) -> dict:
    """Verify every split contains >= 1 sample of every required class before training."""
    summary = {}
    for sp in ("train", "val", "test"):
        recs = split_records.get(sp, [])
        counts = Counter(int(r["label"]) for r in recs)
        missing = [class_names[i] for i in range(len(class_names)) if counts.get(i, 0) == 0]
        if missing:
            raise RuntimeError(
                f"Split '{sp}' is missing required classes {missing} (counts={dict(counts)}). "
                "Cannot compute valid class weights or Macro-F1 without all classes."
            )
        summary[sp] = {class_names[i]: counts[i] for i in range(len(class_names))}
    return summary

SPLIT_CLASS_SUMMARY = validate_split_classes(records_by_split, CONFIG["class_names"])
for sp, cnt in SPLIT_CLASS_SUMMARY.items():
    print(f"{sp:>5}: total={sum(cnt.values()):>6} | {cnt}")

MEAN = tuple(CONFIG["normalization"]["mean"])
STD = tuple(CONFIG["normalization"]["std"])
INPUT_SIZE = CONFIG["input_size"]
RESIZE_EVAL = int(round(INPUT_SIZE * 1.143))

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.82, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(RESIZE_EVAL),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class StreamingHumanFaceDataset(Dataset):
    def __init__(self, records, data_root: Path, transform):
        self.records = records
        self.data_root = data_root
        self.transform = transform
        self.decode_errors = []

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        p = Path(rec["image_path"])
        if not p.is_absolute():
            p = self.data_root / p
        try:
            with Image.open(p) as img:
                img.load()
                tensor = self.transform(img.convert("RGB"))
            return tensor, int(rec["label"]), int(idx)
        except Exception as exc:
            logger.warning("Skipping unreadable image '%s': %s", p, exc)
            self.decode_errors.append((str(p), str(exc)))
            return None

def safe_collate_valid_samples(batch):
    """Filter out unreadable images cleanly without fabricating tensors or labels."""
    valid = [item for item in batch if item is not None]
    if not valid:
        return None
    imgs = torch.stack([v[0] for v in valid], dim=0)
    labels = torch.tensor([v[1] for v in valid], dtype=torch.long)
    indices = [v[2] for v in valid]
    return imgs, labels, indices

train_ds = StreamingHumanFaceDataset(records_by_split["train"], DATA_ROOT, train_transform)
val_ds = StreamingHumanFaceDataset(records_by_split["val"], DATA_ROOT, eval_transform)
test_ds = StreamingHumanFaceDataset(records_by_split["test"], DATA_ROOT, eval_transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True, collate_fn=safe_collate_valid_samples)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True, collate_fn=safe_collate_valid_samples)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True, collate_fn=safe_collate_valid_samples)

train_counts = Counter(int(r["label"]) for r in records_by_split["train"])
total_train = sum(train_counts.values())
class_weights = torch.tensor(
    [total_train / (CONFIG["num_classes"] * train_counts[i]) for i in range(CONFIG["num_classes"])],
    dtype=torch.float32,
    device=DEVICE,
)
print(f"Class weights: {class_weights.tolist()}")

## Step 6: Build EfficientNet-B0 (ImageNet-Pretrained, 2-Class Head)
Preserves the `model.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, 2))` architecture.
Keeps `model.head` `Dropout` active during Stage 1 warmup while freezing the backbone in `.eval()`.

In [ ]:
import torch.nn as nn

SELECTIVE_UNFREEZE_PREFIXES = ("blocks.5.", "blocks.6.", "conv_head.", "bn2.", "head.")

def build_model(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(CONFIG["backbone"], pretrained=pretrained, num_classes=0)
    num_features = int(model.num_features)
    model.head = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(num_features, CONFIG["num_classes"]),
    )

    def _patched_forward(x):
        feats = model.forward_features(x)
        pooled = model.forward_head(feats, pre_logits=True)
        return model.head(pooled)

    model.forward = _patched_forward
    return model

def configure_stage_trainable(model: nn.Module, stage: int) -> None:
    """Stage 1: head only. Stage 2: selective upper blocks + head."""
    for name, param in model.named_parameters():
        if stage == 1:
            param.requires_grad = name.startswith("head.")
        elif stage == 2:
            param.requires_grad = any(name.startswith(prefix) for prefix in SELECTIVE_UNFREEZE_PREFIXES)
        else:
            raise ValueError(f"Unknown stage: {stage}")

def set_stage_train_mode(model: nn.Module, stage: int) -> None:
    """
    Set module train/eval modes per stage:
    - Stage 1: freeze backbone submodules in eval() while KEEPING `model.head` (including Dropout) in train()!
    - Stage 2: freeze earlier backbone BatchNorm layers in eval() while keeping unfrozen blocks & head in train().
    """
    model.train()
    for name, module in model.named_modules():
        if name == "":
            continue
        if stage == 1:
            if not (name == "head" or name.startswith("head.")):
                module.eval()
        elif stage == 2:
            if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                if not any(name.startswith(p.rstrip(".")) for p in SELECTIVE_UNFREEZE_PREFIXES):
                    module.eval()
    model.head.train()

model = build_model(pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Initialized {CONFIG['backbone']} (2-Class Head) | total_params={total_params:,} | trainable={trainable_params:,}")

## Step 7: Binary Evaluation Utilities (Accuracy, Macro-F1, Precision, Recall, Confusion Matrix, ROC-AUC)
Evaluates batches and validation splits with strict metric computations.

In [ ]:
from collections import defaultdict

def compute_binary_roc_auc(y_true: np.ndarray, y_scores: np.ndarray):
    """Compute binary ROC-AUC using non-parametric Wilcoxon-Mann-Whitney rank sum."""
    pos_mask = y_true == 1
    n_pos = int(np.sum(pos_mask))
    n_neg = int(len(y_true) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return None
    order = np.argsort(y_scores, kind="mergesort")
    sorted_scores = y_scores[order]
    sorted_pos = pos_mask[order]
    ranks = np.empty(len(y_scores), dtype=np.float64)
    i, n = 0, len(sorted_scores)
    while i < n:
        j = i + 1
        while j < n and sorted_scores[j] == sorted_scores[i]:
            j += 1
        ranks[i:j] = 0.5 * ((i + 1) + j)
        i = j
    u_stat = float(np.sum(ranks[sorted_pos])) - (n_pos * (n_pos + 1)) / 2.0
    return float(u_stat / (n_pos * n_neg))

def compute_multiclass_metrics(y_true, y_probs: np.ndarray, require_all_classes: bool = True):
    class_names = CONFIG["class_names"]
    y_true_arr = np.asarray(y_true, dtype=np.int64)
    y_pred_arr = np.argmax(y_probs, axis=1)
    c_cnt = len(class_names)
    cm = np.zeros((c_cnt, c_cnt), dtype=np.int64)
    for t, p in zip(y_true_arr, y_pred_arr):
        cm[int(t), int(p)] += 1
    total = int(len(y_true_arr))
    acc = float(np.trace(cm) / total)
    per_class = {}
    f1s, aucs, missing_classes = [], [], []
    for idx, cls_name in enumerate(class_names):
        tp = int(cm[idx, idx])
        fn = int(np.sum(cm[idx, :]) - tp)
        fp = int(np.sum(cm[:, idx]) - tp)
        tn = int(total - tp - fn - fp)
        support = tp + fn
        prec = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
        rec = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
        f1 = float((2 * prec * rec) / (prec + rec)) if (prec + rec) > 0 else 0.0
        fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
        fnr = float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0
        auc = compute_binary_roc_auc((y_true_arr == idx).astype(np.int64), y_probs[:, idx])
        if auc is not None:
            aucs.append(auc)
        if support == 0:
            missing_classes.append(cls_name)
        else:
            f1s.append(f1)
        per_class[cls_name] = {
            "support": support,
            "precision": round(prec, 6),
            "recall": round(rec, 6),
            "f1": round(f1, 6),
            "fpr": round(fpr, 6),
            "fnr": round(fnr, 6),
            "ovr_roc_auc": round(auc, 6) if auc is not None else None,
        }
    if missing_classes and require_all_classes:
        raise ValueError(f"Evaluation batch/split missing required classes: {missing_classes}")
    non_genuine = y_true_arr != 0
    fgr = float(np.mean(y_pred_arr[non_genuine] == 0)) if np.any(non_genuine) else None
    return {
        "total_samples": total,
        "accuracy": round(acc, 6),
        "macro_f1": round(float(np.mean(f1s)) if f1s else 0.0, 6),
        "macro_ovr_roc_auc": round(float(np.mean(aucs)) if aucs else 0.0, 6) if aucs else None,
        "false_genuine_rate": round(fgr, 6) if fgr is not None else None,
        "missing_classes": missing_classes,
        "confusion_matrix": cm.tolist(),
        "per_class": per_class,
    }

@torch.no_grad()
def evaluate_loader(model: nn.Module, loader: DataLoader, criterion: nn.Module):
    model.eval()
    losses = []
    probs_list, targets_list, indices_list = [], [], []
    for batch in loader:
        if batch is None:
            continue
        images, targets, batch_indices = batch
        images = images.to(DEVICE, non_blocking=True)
        targets_dev = targets.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"] and DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets_dev)
        losses.append(float(loss.item()))
        probs_list.append(torch.softmax(logits, dim=1).cpu().numpy())
        targets_list.extend(targets.numpy().tolist())
        indices_list.extend(batch_indices)
    if not probs_list:
        raise RuntimeError("Evaluation DataLoader yielded zero valid samples.")
    y_probs = np.concatenate(probs_list, axis=0)
    metrics = compute_multiclass_metrics(targets_list, y_probs, require_all_classes=True)
    metrics["loss"] = round(float(np.mean(losses)), 6)
    return metrics, targets_list, y_probs, indices_list

## Step 8: Two-Stage Fine-Tuning with AMP & Macro-F1 Checkpoint Selection
Runs Stage 1 (head warmup) and Stage 2 (selective unfreeze). Automatically saves checkpoints and training history directly to Google Drive.

In [ ]:
import datetime

RESUME_PATH = OUTPUT_DIR / "resume_checkpoint.pt"
BEST_DEPLOY_PATH = OUTPUT_DIR / "human_face_efficientnet_b0_best.pt"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG["label_smoothing"])
scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["use_amp"] and DEVICE.type == "cuda")

total_epochs = CONFIG["stage1_epochs"] + CONFIG["stage2_epochs"]
start_epoch = 1
best_macro_f1 = -1.0
no_improve_count = 0
history = []

def make_optimizer(stage: int):
    configure_stage_trainable(model, stage=stage)
    if stage == 1:
        head_params = [p for _, p in model.named_parameters() if p.requires_grad]
        return torch.optim.AdamW(head_params, lr=CONFIG["stage1_lr"], weight_decay=CONFIG["weight_decay"])
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head.")]
    return torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": CONFIG["stage2_backbone_lr"]},
            {"params": head_params, "lr": CONFIG["stage2_head_lr"]},
        ],
        weight_decay=CONFIG["weight_decay"],
    )

def restore_training_state(resume_path: Path, model_obj: nn.Module, scaler_obj):
    """Restore epoch, stage, best_macro_f1, no_improve_count, history, optimizer, and scaler."""
    ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
    model_obj.load_state_dict(ckpt["model_state_dict"])
    next_ep = int(ckpt["epoch"]) + 1
    prev_stage = int(ckpt.get("stage", 1))
    tgt_stage = 1 if next_ep <= CONFIG["stage1_epochs"] else 2
    restored_patience = 0 if tgt_stage != prev_stage else int(ckpt.get("no_improve_count", 0))
    opt = make_optimizer(stage=tgt_stage)
    if prev_stage == tgt_stage and "optimizer_state_dict" in ckpt:
        opt.load_state_dict(ckpt["optimizer_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler_obj.load_state_dict(ckpt["scaler_state_dict"])
    return (
        next_ep,
        tgt_stage,
        float(ckpt["best_macro_f1"]),
        restored_patience,
        list(ckpt.get("history", [])),
        opt,
    )

current_stage = 1
optimizer = make_optimizer(stage=1)

if CONFIG["resume_if_available"] and RESUME_PATH.exists():
    start_epoch, current_stage, best_macro_f1, no_improve_count, history, optimizer = restore_training_state(
        RESUME_PATH, model, scaler
    )
    print(
        f"Resumed from epoch {start_epoch - 1} | stage={current_stage} | "
        f"best_macro_f1={best_macro_f1:.4f} | no_improve_count={no_improve_count}"
    )

for epoch in range(start_epoch, total_epochs + 1):
    target_stage = 1 if epoch <= CONFIG["stage1_epochs"] else 2
    if target_stage != current_stage:
        current_stage = target_stage
        optimizer = make_optimizer(stage=current_stage)
        no_improve_count = 0
        print(f"--- Transitioned to Stage {current_stage} at epoch {epoch} (patience reset) ---")

    set_stage_train_mode(model, stage=current_stage)
    train_losses = []
    for batch in train_loader:
        if batch is None:
            continue
        images, targets, _ = batch
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CONFIG["use_amp"] and DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(float(loss.item()))

    if not train_losses:
        raise RuntimeError(f"Epoch {epoch}: zero valid training batches processed.")

    total_corrupt = len(train_ds.decode_errors) + len(val_ds.decode_errors)
    if total_corrupt > max(10, int(CONFIG["max_corrupt_fraction"] * len(train_ds))):
        raise RuntimeError(f"Exceeded corrupt-image tolerance ({total_corrupt} decode errors).")

    val_metrics, _, _, _ = evaluate_loader(model, val_loader, criterion)
    mean_train_loss = round(float(np.mean(train_losses)), 6)
    epoch_entry = {
        "epoch": epoch,
        "stage": current_stage,
        "train_loss": mean_train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_macro_ovr_roc_auc": val_metrics["macro_ovr_roc_auc"],
    }
    history.append(epoch_entry)
    print(
        f"Epoch {epoch:02d}/{total_epochs} (S{current_stage}) | "
        f"train_loss={mean_train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        no_improve_count = 0
        deploy_payload = {
            "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "backbone": CONFIG["backbone"],
            "num_classes": CONFIG["num_classes"],
            "class_names": list(CONFIG["class_names"]),
            "class_to_id": dict(CONFIG["class_to_id"]),
            "input_size": CONFIG["input_size"],
            "normalization": dict(CONFIG["normalization"]),
            "model_version": CONFIG["model_version"],
            "architecture": "EfficientNet-B0 transfer learning (ImageNet initialization, 2-class head: Dropout(0.2) + Linear(1280, 2))",
            "best_val_metrics": val_metrics,
            "training_config": dict(CONFIG),
            "epoch": epoch,
            "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }
        torch.save(deploy_payload, BEST_DEPLOY_PATH)
        print(f"  -> Saved NEW best deployment checkpoint (Macro-F1={best_macro_f1:.4f}) to {BEST_DEPLOY_PATH.name}")
    else:
        no_improve_count += 1

    torch.save(
        {
            "epoch": epoch,
            "stage": current_stage,
            "best_macro_f1": best_macro_f1,
            "no_improve_count": no_improve_count,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
        },
        RESUME_PATH,
    )
    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    if current_stage == 2 and no_improve_count >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping triggered after {no_improve_count} Stage-2 epochs without Macro-F1 improvement.")
        break

## Step 9: Final Test-Set Evaluation, Visualizations & Deployment Report
Evaluates the best checkpoint on the held-out test split (14,000 images: 7,000 genuine + 7,000 AI-generated), plots training/validation loss curves, confusion matrix, ROC-AUC curve, and saves evaluation reports to Google Drive.

In [ ]:
import matplotlib.pyplot as plt

def compute_per_generator_breakdown_colab(records, valid_indices, y_true, y_probs, class_names):
    preds = np.argmax(y_probs, axis=1)
    by_gen = defaultdict(list)
    for pos, rec_idx in enumerate(valid_indices):
        by_gen[records[rec_idx].get("source_subset", "unknown")].append(pos)
    breakdown = {}
    for gen_name, pos_idxs in sorted(by_gen.items()):
        sub_t = np.asarray([y_true[i] for i in pos_idxs], dtype=np.int64)
        sub_p = preds[pos_idxs]
        dist_raw = Counter(sub_t.tolist())
        label_dist = {class_names[int(k)]: int(v) for k, v in sorted(dist_raw.items())}
        expected_cls = class_names[int(sub_t[0])] if len(label_dist) == 1 else "mixed"
        non_genuine_mask = sub_t != 0
        fgr = float(np.mean(sub_p[non_genuine_mask] == 0)) if np.any(non_genuine_mask) else None
        breakdown[gen_name] = {
            "count": len(pos_idxs),
            "expected_class": expected_cls,
            "label_distribution": label_dist,
            "non_genuine_count": int(np.sum(non_genuine_mask)),
            "accuracy": round(float(np.mean(sub_p == sub_t)), 6),
            "false_genuine_rate": round(fgr, 6) if fgr is not None else None,
        }
    return breakdown

best_ckpt = torch.load(BEST_DEPLOY_PATH, map_location=DEVICE, weights_only=False)
assert "optimizer_state_dict" not in best_ckpt, "Deployment checkpoint must NOT contain optimizer_state_dict."
assert list(best_ckpt["class_names"]) == list(CONFIG["class_names"]), "Checkpoint class ordering mismatch!"

eval_model = build_model(pretrained=False).to(DEVICE)
eval_model.load_state_dict(best_ckpt["model_state_dict"], strict=True)

test_metrics, test_targets, test_probs, test_valid_indices = evaluate_loader(eval_model, test_loader, criterion)
gen_breakdown = compute_per_generator_breakdown_colab(
    records_by_split["test"], test_valid_indices, test_targets, test_probs, CONFIG["class_names"]
)

report = {
    "evaluated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_version": best_ckpt["model_version"],
    "best_epoch": best_ckpt["epoch"],
    "test_metrics": test_metrics,
    "per_generator_breakdown": gen_breakdown,
}

REPORT_JSON_PATH = OUTPUT_DIR / "test_evaluation_report.json"
with open(REPORT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

# --- VISUALIZATION DASHBOARD ---
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Plot 1: Loss curves
epochs_range = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
axes[0, 0].plot(epochs_range, train_losses, marker="o", label="Train Loss", color="#1f77b4")
axes[0, 0].plot(epochs_range, val_losses, marker="s", label="Val Loss", color="#ff7f0e")
axes[0, 0].axvline(x=CONFIG["stage1_epochs"] + 0.5, color="gray", linestyle="--", label="Stage 1->2 Transition")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("CrossEntropy Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Validation Metrics
val_f1 = [h["val_macro_f1"] for h in history]
val_acc = [h["val_accuracy"] for h in history]
axes[0, 1].plot(epochs_range, val_f1, marker="^", label="Val Macro-F1", color="#2ca02c")
axes[0, 1].plot(epochs_range, val_acc, marker="d", label="Val Accuracy", color="#d62728")
axes[0, 1].axvline(x=best_ckpt["epoch"], color="green", linestyle=":", label=f"Best Model (Ep {best_ckpt['epoch']})")
axes[0, 1].set_title("Validation Accuracy & Macro-F1")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Score")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Confusion Matrix Heatmap
cm = np.array(test_metrics["confusion_matrix"])
cax = axes[1, 0].matshow(cm, cmap="Blues")
fig.colorbar(cax, ax=axes[1, 0])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1, 0].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black" if cm[i, j] < cm.max()/2 else "white")
axes[1, 0].set_xticks(range(len(CONFIG["class_names"])))
axes[1, 0].set_yticks(range(len(CONFIG["class_names"])))
axes[1, 0].set_xticklabels(CONFIG["class_names"])
axes[1, 0].set_yticklabels(CONFIG["class_names"])
axes[1, 0].set_title(f"Test Confusion Matrix (Total={test_metrics['total_samples']:,})")
axes[1, 0].set_xlabel("Predicted Label")
axes[1, 0].set_ylabel("True Label")

# Plot 4: Per-Class Precision & Recall
cls_names = CONFIG["class_names"]
precisions = [test_metrics["per_class"][c]["precision"] for c in cls_names]
recalls = [test_metrics["per_class"][c]["recall"] for c in cls_names]
x = np.arange(len(cls_names))
w = 0.35
axes[1, 1].bar(x - w/2, precisions, w, label="Precision", color="#4a90e2")
axes[1, 1].bar(x + w/2, recalls, w, label="Recall", color="#50e3c2")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(cls_names)
axes[1, 1].set_ylim(0.0, 1.05)
axes[1, 1].set_title(f"Test Precision & Recall (Macro-F1={test_metrics['macro_f1']:.4f})")
axes[1, 1].set_ylabel("Score")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
DASHBOARD_PATH = OUTPUT_DIR / "training_evaluation_dashboard.png"
plt.savefig(DASHBOARD_PATH, dpi=200)
plt.show()

print("\n=== FINAL TEST EVALUATION REPORT ===")
print(f"Total Test Samples:  {test_metrics['total_samples']:,}")
print(f"Test Accuracy:       {test_metrics['accuracy'] * 100:.2f}%")
print(f"Test Macro-F1:       {test_metrics['macro_f1']:.4f}")
print(f"Test ROC-AUC:        {test_metrics['macro_ovr_roc_auc']:.4f}")
if test_metrics["false_genuine_rate"] is not None:
    print(f"False-Genuine Rate:  {test_metrics['false_genuine_rate'] * 100:.2f}%")
for c in CONFIG["class_names"]:
    m = test_metrics["per_class"][c]
    print(f" - {c:<12}: Precision={m['precision']:.4f} | Recall={m['recall']:.4f} | F1={m['f1']:.4f} | Support={m['support']:,}")
print(f"\nSaved dashboard to: {DASHBOARD_PATH}")
print(f"Saved evaluation report to: {REPORT_JSON_PATH}")
print(f"Saved best deployment checkpoint to: {BEST_DEPLOY_PATH}")